# nb30 - Subtract-then-calibrate: making the fraction head load-bearing

nb29 set the ceiling: PERFECT per-cell fractions at 5x5 give sigma_eff ~0.032 (per-bin 0.025-0.035) - right at the goal - while binary keep/drop only reaches ~0.055. nb28 showed multitask fraction supervision alone does not move the energy head (0.0502 -> 0.0512): the pooled-MLP readout is free to ignore f-hat. **v2 wires the estimator into the architecture**: E = calib(sum_i sigmoid(f_i) * e_i) + residual(pooled, globals), so the gradient of the energy loss flows THROUGH the subtracted sum and the fraction head must earn its keep. Domain-adversarial variant (GRL) closes the overlay-vs-real gap (nb28 closure AUC ~0.74).

Configs: `sub_nosup` (subtraction readout, no frac labels - is structure alone enough?), `sub_frac` (+ per-cell BCE, lambda 1.0), `sub_frac_dann` (+ gradient-reversal domain head). All train on real minbias + overlays; early-stop and recalibration on real val; test on the real min-bias test split. Anchors: base_real 0.0502 (nb28), kNN-81 0.0459 (nb22), oracle 0.032 (nb29), goal 0.03.

In [1]:
import os, sys, glob, copy, time, pathlib
import numpy as np, pandas as pd, uproot, awkward as ak, matplotlib.pyplot as plt
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
CLEAN = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
MB = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
OUT = REPO / 'reports' / 'predictions'; OUT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB30_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB30_MODE', 'full')
if MODE == 'smoke': CLEAN, MB = CLEAN[:4], MB[:8]
THRESH = 2.49
W = 2; L = (2*W+1)**2
print('device', DEVICE, '| mode', MODE, '|', len(CLEAN), 'clean,', len(MB), 'minbias | THRESH', THRESH, 'MeV')

device cuda | mode full | 100 clean, 94 minbias | THRESH 2.49 MeV


In [2]:
TK = ['cell_x','cell_y','energy','cell_energies_front','cell_energies_back',
      'cell_times_front','cell_times_back','imodx','jmody']
AUX = ['sig_flux_prod_vertex_z','sig_flux_eTot']
def event_geom(cc):
    x, yy, e = cc['cell_x'], cc['cell_y'], cc['energy']
    ix, iy = cc['imodx'], cc['jmody']
    seed = int(np.argmax(e))
    pts = np.stack([x, yy], 1)
    pitch = np.full(len(x), np.nan)
    for key in {(int(p), int(q)) for p, q in zip(ix, iy)}:
        sel = (ix == key[0]) & (iy == key[1]); p = pts[sel]
        if len(p) >= 2:
            d = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf
            pitch[sel] = np.median(np.min(d, axis=1))
    fill = np.nanmedian(pitch) if np.isfinite(pitch).any() else 120.0
    pitch[~np.isfinite(pitch)] = fill
    ps = pitch[seed]
    ei = (x - x[seed]) / ps; ej = (yy - yy[seed]) / ps
    di = np.round(ei).astype(int); dj = np.round(ej).astype(int)
    ok = (np.abs(ei - di) < 0.15) & (np.abs(ej - dj) < 0.15)
    return seed, ps, di, dj, ok
def build_grid(files, keep_cheby, label):
    EV = []
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(TK + AUX, library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        et_all = ak.to_numpy(a['sig_flux_eTot']).astype(float)
        for i in np.flatnonzero((vz < 100.0) & (et_all >= 1.0) & (et_all <= 100.0)):
            cc = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TK}
            e = cc['energy']
            if len(e) < 3: continue
            seed, ps, di, dj, ok = event_geom(cc)
            if ok.mean() < 0.5: continue
            ch = np.maximum(np.abs(di), np.abs(dj))
            keep = ok if keep_cheby is None else (ok & (ch <= keep_cheby))
            if keep.sum() < 1 or not keep[seed]: continue
            tf = cc['cell_times_front'][keep]; tb = cc['cell_times_back'][keep]
            tf = np.where(np.isfinite(tf) & (tf != 0) & (np.abs(tf) < 1e4), tf, np.nan)
            tb = np.where(np.isfinite(tb) & (tb != 0) & (np.abs(tb) < 1e4), tb, np.nan)
            EV.append(dict(di=di[keep].astype(np.int16), dj=dj[keep].astype(np.int16),
                           e=e[keep].astype(np.float32),
                           fr=cc['cell_energies_front'][keep].astype(np.float32),
                           bk=cc['cell_energies_back'][keep].astype(np.float32),
                           tf=tf.astype(np.float32), tb=tb.astype(np.float32),
                           ps=float(ps), reg=int(np.argmin(np.abs(PITCH - ps))),
                           rmax=int(ch[ok].max()), Etrue=float(et_all[i])))
    print(f'{label}: {len(EV)} events')
    return EV
t0 = time.time()
CE = build_grid(CLEAN, 3, 'clean')
ME = build_grid(MB, None, 'minbias')
print(f'build {time.time()-t0:.0f}s')

clean: 30303 events


minbias: 72554 events
build 504s


In [3]:
def window_tokens(di, dj, e, fr, bk, tf, tb, ps, reg, frac=None):
    m = (np.maximum(np.abs(di), np.abs(dj)) <= W) & (e >= THRESH)
    if m.sum() < 1: return None
    di, dj, e, fr, bk, tf, tb = (v[m] for v in (di, dj, e, fr, bk, tf, tb))
    t0f = np.nanmedian(tf) if np.isfinite(tf).any() else 0.0
    t0b = np.nanmedian(tb) if np.isfinite(tb).any() else 0.0
    tfc = np.where(np.isfinite(tf), tf - t0f, 0.0); htf = np.isfinite(tf).astype(np.float32)
    tbc = np.where(np.isfinite(tb), tb - t0b, 0.0); htb = np.isfinite(tb).astype(np.float32)
    rdr = np.hypot(di, dj)
    cont = np.stack([np.log1p(np.clip(e, 0, None)), np.log1p(np.clip(fr, 0, None)),
                     np.log1p(np.clip(bk, 0, None)), di.astype(np.float32), dj.astype(np.float32),
                     rdr, np.full(len(e), np.log(ps)), np.clip(tfc, -5, 5), np.clip(tbc, -5, 5)], 1)
    oh = np.zeros((len(e), len(PITCH)), np.float32); oh[:, reg] = 1.0
    tok = np.concatenate([cont, htf[:, None], htb[:, None], oh], 1).astype(np.float32)
    fout = None if frac is None else frac[m].astype(np.float32)
    return tok, float(e.sum()), float(e.max()), fout
def real_window(ev):
    r = window_tokens(ev['di'], ev['dj'], ev['e'], ev['fr'], ev['bk'], ev['tf'], ev['tb'], ev['ps'], ev['reg'])
    return None if r is None else (r[0], r[1], r[2], None, ev['Etrue'])
def add_donor(pos, B, rng):
    di0, dj0 = 0, 0
    while max(abs(di0), abs(dj0)) < 4:
        di0 = int(rng.integers(-5, 6)); dj0 = int(rng.integers(-5, 6))
    for k in range(len(B['e'])):
        ri, rj = int(B['di'][k]) - di0, int(B['dj'][k]) - dj0
        if max(abs(ri), abs(rj)) > W: continue
        eb = float(B['e'][k]); tfb = float(B['tf'][k]); tbb = float(B['tb'][k])
        if (ri, rj) in pos:
            p = pos[(ri, rj)]
            for idx, val in ((3, tfb), (4, tbb)):
                if np.isfinite(val) and np.isfinite(p[idx]):
                    p[idx] = (p[idx] * p[0] + val * eb) / (p[0] + eb)
                elif np.isfinite(val): p[idx] = val
            p[0] += eb; p[1] += float(B['fr'][k]); p[2] += float(B['bk'][k]); p[6] += eb
        else:
            pos[(ri, rj)] = [eb, float(B['fr'][k]), float(B['bk'][k]), tfb, tbb, 0.0, eb]
def overlay(A, pool, rng, target_ratio):
    pos = {}
    for k in range(len(A['e'])):
        if max(abs(A['di'][k]), abs(A['dj'][k])) <= W:
            pos[(int(A['di'][k]), int(A['dj'][k]))] = [float(A['e'][k]), float(A['fr'][k]), float(A['bk'][k]),
                                                       float(A['tf'][k]), float(A['tb'][k]), float(A['e'][k]), 0.0]
    if not pos: return None
    et_mev = A['Etrue'] * 1e3
    nd = 0
    while nd < 4:
        if sum(p[0] for p in pos.values()) / et_mev >= target_ratio: break
        add_donor(pos, pool[int(rng.integers(len(pool)))], rng); nd += 1
    keys = list(pos.keys()); V = np.array([pos[k] for k in keys], np.float64)
    di = np.array([k[0] for k in keys], np.int16); dj = np.array([k[1] for k in keys], np.int16)
    if not (di[np.argmax(V[:, 0])] == 0 and dj[np.argmax(V[:, 0])] == 0): return None
    frac = V[:, 5] / np.maximum(V[:, 0], 1e-9)
    r = window_tokens(di, dj, V[:, 0], V[:, 1], V[:, 2], V[:, 3], V[:, 4], A['ps'], A['reg'], frac=frac)
    return None if r is None else (r[0], r[1], r[2], r[3], A['Etrue'])

In [4]:
n_real = len(ME)
rtr, rva, rte = split(n_real)
rng = np.random.default_rng(0)
donors_by_reg = {}
for i in rtr:
    ev = ME[i]
    if ev['rmax'] >= 7: donors_by_reg.setdefault(ev['reg'], []).append(ev)
REAL = []; keep_real = []
for i, ev in enumerate(ME):
    r = real_window(ev)
    if r is not None: REAL.append(r); keep_real.append(i)
keep_real = np.array(keep_real)
remap = -np.ones(n_real, int); remap[keep_real] = np.arange(len(REAL))
tr_mask = np.zeros(len(REAL), np.bool_); tr_mask[remap[rtr][remap[rtr] >= 0]] = True
ratio_pool = np.array([REAL[i][1] / (REAL[i][4] * 1e3) for i in np.flatnonzero(tr_mask)])
rtr = remap[rtr][remap[rtr] >= 0]; rva = remap[rva][remap[rva] >= 0]; rte = remap[rte][remap[rte] >= 0]
MAX_OVR = 2000 if MODE == 'smoke' else 60000
OVR = []; drop_seed = 0
for idx in rng.permutation(len(CE)):
    if len(OVR) >= MAX_OVR: break
    A = CE[idx]
    pool = donors_by_reg.get(A['reg'])
    if not pool: continue
    tgt = float(ratio_pool[int(rng.integers(len(ratio_pool)))])
    r = overlay(A, pool, rng, tgt)
    if r is None: drop_seed += 1; continue
    OVR.append(r)
print(f'real {len(REAL)} (tr/va/te {len(rtr)}/{len(rva)}/{len(rte)}) | overlays {len(OVR)} (dropped {drop_seed})')

real 72554 (tr/va/te 50787/10883/10884) | overlays 17302 (dropped 965)


In [5]:
ALL = REAL + OVR
N = len(ALL); IN_DIM = ALL[0][0].shape[1]; NG = 5
n_ova = min(3000, len(OVR) // 10)
otr = np.arange(len(REAL), N - n_ova); ova = np.arange(N - n_ova, N)
y = np.array([np.log(max(r[4], 1e-3)) for r in ALL], np.float32)
Et = np.array([r[4] for r in ALL], np.float32)
sumE = np.array([r[1] for r in ALL], np.float32)
X = np.zeros((N, L, IN_DIM), np.float32); M = np.zeros((N, L), np.bool_)
Fr = np.zeros((N, L), np.float32); HasF = np.zeros(N, np.bool_)
G = np.zeros((N, NG), np.float32); Eraw = np.zeros((N, L), np.float32)
for i, (tok, se, sde, fr_t, et) in enumerate(ALL):
    n = tok.shape[0]; X[i, :n] = tok; M[i, :n] = True
    if fr_t is not None: Fr[i, :n] = fr_t; HasF[i] = True
    e = np.expm1(tok[:, 0]); Eraw[i, :n] = e
    lat = float(np.sqrt((e * tok[:, 5] ** 2).sum() / (e.sum() + EPS)))
    fbr = float(np.expm1(tok[:, 1]).sum() / (np.expm1(tok[:, 2]).sum() + EPS))
    G[i] = [np.log1p(se), np.log1p(sde), np.log(n), fbr, lat]
la0, lb0 = np.polyfit(np.log1p(0.5 * sumE[rtr]), y[rtr], 1)
G = (G - G[rtr].mean(0)) / (G[rtr].std(0) + EPS)
NC = 9
cont = X[rtr][:, :, :NC].reshape(-1, NC)[M[rtr].reshape(-1)]
mean = cont.mean(0); std = cont.std(0) + EPS
X[:, :, :NC] = (X[:, :, :NC] - mean) / std; X[~M] = 0.0
dom = np.zeros(N, np.float32); dom[len(REAL):] = 1.0
Xc = torch.from_numpy(X).to(DEVICE); Mc = torch.from_numpy(M).to(DEVICE)
Frc = torch.from_numpy(Fr).to(DEVICE); HFc = torch.from_numpy(HasF).to(DEVICE)
Gc = torch.from_numpy(G).to(DEVICE); Yc = torch.from_numpy(y).unsqueeze(1).to(DEVICE)
Ec = torch.from_numpy(Eraw).to(DEVICE); Dc = torch.from_numpy(dom).unsqueeze(1).to(DEVICE)
print('tensors N', N, '(real', len(REAL), '+ ovr-train', len(otr), '+ ovr-holdout', n_ova, ') on', DEVICE)

tensors N 89856 (real 72554 + ovr-train 15572 + ovr-holdout 1730 ) on cuda


In [6]:
CFG = dict(d=96, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96, huber_delta=0.1, lam=1.0, lam_d=0.2)
class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, l):
        ctx.l = l; return x.view_as(x)
    @staticmethod
    def backward(ctx, g):
        return -ctx.l * g, None
class SubFrac(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        d = CFG['d']
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                           dropout=CFG['dropout'], batch_first=True)
        self.enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 1))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.dhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell, grl=0.0):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        fl = self.fhead(h).squeeze(-1)
        w = torch.sigmoid(fl) * m.float()
        s_sub = (w * ecell).sum(1, keepdim=True)
        base = self.la * torch.log1p(s_sub) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        dlog = self.dhead(GRL.apply(p, grl))
        return base + self.head(torch.cat([p, g], 1)), fl, dlog
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
def train_eval(config, seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubFrac(IN_DIM).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ck = CKPT / f'nb30_{config}_s{seed}.pt'
    tr_idx = np.concatenate([np.asarray(rtr), otr])
    lam = 0.0 if config == 'sub_nosup' else CFG['lam']
    dann = config.endswith('dann')
    def lossf(b, grl):
        pe, fl, dlog = model(Xc[b], Mc[b], Gc[b], Ec[b], grl)
        loss = nn.functional.huber_loss(pe, Yc[b], delta=CFG['huber_delta'])
        if lam > 0:
            hb = HFc[b]
            if hb.any():
                wq = Ec[b][hb] * Mc[b][hb]
                wq = wq / wq.sum().clamp(min=1e-9)
                bce = nn.functional.binary_cross_entropy_with_logits(fl[hb], Frc[b][hb], reduction='none')
                loss = loss + lam * (bce * wq).sum()
        if dann and grl > 0:
            loss = loss + nn.functional.binary_cross_entropy_with_logits(dlog, Dc[b])
        return loss
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def run(idx):
        model.eval(); out = []
        with torch.no_grad():
            for b in batches(idx, 256, False): out.append(model(Xc[b], Mc[b], Gc[b], Ec[b])[0].cpu().numpy().ravel())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(rva, 256, False):
                pe, _, _ = model(Xc[b], Mc[b], Gc[b], Ec[b])
                s += nn.functional.huber_loss(pe, Yc[b], delta=CFG['huber_delta']).item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume {config} s{seed} from epoch {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        grl = CFG['lam_d'] * min(1.0, ep / 10.0) if dann else 0.0
        for b in batches(tr_idx, CFG['batch'], True):
            opt.zero_grad(); lossf(b, grl).backward(); opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    model.load_state_dict(bstate)
    a, b2 = np.polyfit(run(rva), y[rva], 1)
    pe = np.exp(a * run(rte) + b2)
    model.eval()
    with torch.no_grad():
        fm = []; sub_r = []
        for b in [torch.from_numpy(np.asarray(ova)).to(DEVICE)]:
            _, fl, _ = model(Xc[b], Mc[b], Gc[b], Ec[b])
            fh = torch.sigmoid(fl); wq = Ec[b] * Mc[b]
            fm = (torch.abs(fh - Frc[b]) * wq).sum() / wq.sum()
        for b in batches(rte, 512, False):
            _, fl, _ = model(Xc[b], Mc[b], Gc[b], Ec[b])
            fh = torch.sigmoid(fl)
            sub_r.append(((fh * Ec[b]).sum(1) / Ec[b].sum(1).clamp(min=1e-9)).cpu().numpy())
    return (float(resolution(pe, Et[rte])['sigma_eff']), pe,
            float(fm), float(np.median(np.concatenate(sub_r))))

In [7]:
EPOCHS = {'smoke': 2, 'full': 60}[MODE]
PATIENCE = {'smoke': 99, 'full': 10}[MODE]
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
CONFIGS = ['sub_frac', 'sub_frac_dann', 'sub_nosup']
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb30_subfrac{TAG}.csv'
done = set()
if CSVP.exists():
    prev = pd.read_csv(CSVP); done = set(zip(prev['config'], prev['seed']))
    print('resume, done:', sorted(done))
for seed in SEEDS:
    for config in CONFIGS:
        if (config, seed) in done: print('skip', config, seed); continue
        t0 = time.time()
        sig, pe, fmae, submed = train_eval(config, seed, EPOCHS, PATIENCE)
        np.save(OUT / f'nb30_pred{TAG}_{config}_s{seed}.npy', pe)
        row = dict(config=config, seed=seed, sigma_eff=round(sig, 4), frac_mae=round(fmae, 4),
                   real_subfrac_med=round(submed, 3), elapsed=round(time.time()-t0))
        pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
        print(f'{config} seed {seed}: sigma_eff {sig:.4f} | frac MAE (ovr holdout) {fmae:.4f} | '
              f'median subtracted-frac on real test {submed:.3f} ({row["elapsed"]}s)', flush=True)
RES = pd.read_csv(CSVP); print(RES.to_string(index=False))

sub_frac seed 0: sigma_eff 0.0485 | frac MAE (ovr holdout) 0.0433 | median subtracted-frac on real test 0.773 (1570s)


sub_frac_dann seed 0: sigma_eff 0.0535 | frac MAE (ovr holdout) 0.0446 | median subtracted-frac on real test 0.776 (1702s)


sub_nosup seed 0: sigma_eff 0.0472 | frac MAE (ovr holdout) 0.2089 | median subtracted-frac on real test 0.548 (1434s)


sub_frac seed 1: sigma_eff 0.0505 | frac MAE (ovr holdout) 0.0483 | median subtracted-frac on real test 0.774 (1355s)


sub_frac_dann seed 1: sigma_eff 0.0524 | frac MAE (ovr holdout) 0.0436 | median subtracted-frac on real test 0.781 (1589s)


sub_nosup seed 1: sigma_eff 0.0490 | frac MAE (ovr holdout) 0.2079 | median subtracted-frac on real test 0.556 (304s)


       config  seed  sigma_eff  frac_mae  real_subfrac_med  elapsed
     sub_frac     0     0.0485    0.0433             0.773     1570
sub_frac_dann     0     0.0535    0.0446             0.776     1702
    sub_nosup     0     0.0472    0.2089             0.548     1434
     sub_frac     1     0.0505    0.0483             0.774     1355
sub_frac_dann     1     0.0524    0.0436             0.781     1589
    sub_nosup     1     0.0490    0.2079             0.556      304


## Verdict
`frac_mae` (energy-weighted, overlay holdout) says whether the fraction head learned its job; `real_subfrac_med` says how much energy it removes on real windows (real median contamination is ~1.19, so a well-transferred head should keep ~0.84 of the window sum). sigma_eff on the real test split against: base_real 0.0502, kNN-81 0.0459, oracle 0.032, goal 0.03.

In [8]:
print('anchors: base_real 0.0502 | kNN-81 0.0459 | oracle-frac 5x5 0.032 | goal 0.03')
for config in CONFIGS:
    sub = RES[RES.config == config]
    if len(sub): print(f'{config:14s}: {sub.sigma_eff.mean():.4f} +/- {(sub.sigma_eff.std() if len(sub) > 1 else 0):.4f} '
                       f'| frac_mae {sub.frac_mae.mean():.4f} | subfrac {sub.real_subfrac_med.mean():.3f}')
npy = {(cfg, s): np.load(OUT / f'nb30_pred{TAG}_{cfg}_s{s}.npy') for cfg in CONFIGS for s in RES[RES.config==cfg].seed
       if (OUT / f'nb30_pred{TAG}_{cfg}_s{s}.npy').exists()}
best_cfg = RES.groupby('config').sigma_eff.mean().idxmin()
ps = [p for (cfg, s), p in npy.items() if cfg == best_cfg and len(p) == len(rte)]
if len(ps) >= 2:
    print(f'{best_cfg} seed-ensemble: {resolution(np.stack(ps).mean(0), Et[rte])["sigma_eff"]:.4f}')
if ps:
    pe = np.stack(ps).mean(0); te_e = Et[rte]
    edges = np.quantile(te_e, np.linspace(0, 1, 7))
    print(f'per-E-bin sigma_eff ({best_cfg}):')
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        if mm.sum() >= 20:
            print(f'  E {edges[i]:6.1f}-{edges[i+1]:6.1f} GeV: {resolution(pe[mm], te_e[mm])["sigma_eff"]:.4f}  (n={int(mm.sum())})')

anchors: base_real 0.0502 | kNN-81 0.0459 | oracle-frac 5x5 0.032 | goal 0.03
sub_frac      : 0.0495 +/- 0.0014 | frac_mae 0.0458 | subfrac 0.774
sub_frac_dann : 0.0529 +/- 0.0008 | frac_mae 0.0441 | subfrac 0.778
sub_nosup     : 0.0481 +/- 0.0013 | frac_mae 0.2084 | subfrac 0.552
sub_nosup seed-ensemble: 0.0465
per-E-bin sigma_eff (sub_nosup):
  E    2.2-  10.7 GeV: 0.0725  (n=1814)
  E   10.7-  17.4 GeV: 0.0521  (n=1814)
  E   17.4-  24.0 GeV: 0.0409  (n=1814)
  E   24.0-  34.1 GeV: 0.0384  (n=1814)
  E   34.1-  53.1 GeV: 0.0384  (n=1814)
  E   53.1- 100.0 GeV: 0.0403  (n=1814)
